# Verificación — Spark lee tu datalake

**Curso:** ST1630-2026-2 · **Semana:** S4-S5
**Estudiante:** Sebastian Andres Medina Cabezas
**Fecha:** 23/08/2026

## Objetivo

Cerrar el Lab 1a confirmando que tu clúster EMR puede leer el datalake
que construiste (Partes 1-4): conectar Spark a tu bucket S3, leer el
archivo Parquet de Bronze, y repetir el benchmark Parquet vs. CSV visto
en la clase de S4.

**Qué debe verse al final para confirmar que el lab está completo:**
- La Celda 2 muestra el schema y 5 filas del Parquet leído desde S3
  (si esto funciona, tu bucket, tu rol IAM y tu clúster están bien
  configurados de punta a punta).
- La Celda 3 imprime el tiempo de una misma consulta en Parquet y en
  CSV, y el ratio entre ambos.
- Completaste el análisis de la Celda 4 y capturaste el DAG de Spark UI
  como indica la Celda 5.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# getOrCreate() funciona tanto en EMR (donde ya existe una sesión activa
# administrada por el clúster) como en un entorno local con pyspark
# instalado, sin necesitar ramas de código distintas.
spark = SparkSession.builder.appName("ST1630-Lab1a-Verificacion").getOrCreate()

# EDITAR: reemplaza por el bucket que creaste en setup_s3.sh
# (convención: st1630-{tu-usuario}-{año})
BUCKET = "st1630-samedinac-2023"

# En EMR, S3 se referencia directamente con el esquema s3://.
# En local (con las credenciales de AWS Academy exportadas), la misma
# ruta también funciona porque Spark usa el conector S3A por debajo.
ruta_parquet = f"s3://{BUCKET}/bronze/ventas/prueba_parquet.parquet"
ruta_csv = f"s3://{BUCKET}/bronze/ventas/prueba_csv.csv"

df_parquet = spark.read.parquet(ruta_parquet)

df_parquet.printSchema()
df_parquet.show(5, truncate=False)

# Si ves el schema y las filas de arriba, tu datalake funciona
# correctamente de punta a punta: bucket, permisos IAM y clúster EMR.
print("Filas leídas:", df_parquet.count())

No pude crear el Amazon EMR Studio, por lo que tampoco pude crear el Workspace, ya que cuando creab el Studio me decia que no tenia permisos para crearlo serverless, por lo que decidi correr el codigo del notebook en un archivo py dentro del ssh para poder acceder al cluster, por lo que mostrare el resultado copiandolo y mostrando imagenes que demuestren q el codigo verdaderamente corrio

![ejec1](ejecucion1.png)
![ejec2](ejecucion2.png)
![ejec3](ejecucion3.png)
![ejec4](ejecucion4.png)

### Esquema del DataFrame
```text
root
 |-- order_id: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- region: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: long (nullable = true)
 |-- precio_unit: double (nullable = true)
 |-- total: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- devuelto: boolean (nullable = true)
```
### Primeras 5 filas

| order_id | fecha | region | producto | categoria | cantidad | precio_unit | total | canal | devuelto |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| ORD-000001 | 2026-04-22 | Cali | Mouse | Electrónica | 1 | 789300.0 | 789300.0 | online | false |
| ORD-000002 | 2026-03-22 | Barranquilla | Gorra | Ropa | 2 | 57000.0 | 114000.0 | tienda | false |
| ORD-000003 | 2026-01-06 | Bogotá | Zapatos | Ropa | 4 | 163800.0 | 655200.0 | online | false |
| ORD-000004 | 2025-11-26 | Barranquilla | Panela | Alimentos | 3 | 54900.0 | 164700.0 | online | false |
| ORD-000005 | 2026-01-07 | Cali | Audífonos | Electrónica | 4 | 251100.0 | 1004400.0 | tienda | true |

**Mostrando:** Las primeras 5 filas.
**Filas totales:** 10,000



In [ ]:
import time

df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv(ruta_csv)

# Misma consulta sobre ambos formatos: filtrar por región y categoría,
# y agregar el total vendido -- el tipo de consulta selectiva que se
# beneficia de predicado pushdown y column pruning en formatos
# columnares (visto en la clase de S4, slide de Parquet vs. CSV).

def benchmark(df, nombre):
    inicio = time.time()
    resultado = (
        df.filter((F.col("region") == "Bogotá") & (F.col("categoria") == "Electrónica"))
          .groupBy("producto")
          .agg(F.sum("total").alias("total_vendido"))
          .orderBy(F.col("total_vendido").desc())
          .collect()  # acción -- fuerza la ejecución real, no solo el plan
    )
    duracion = time.time() - inicio
    print(f"{nombre}: {duracion:.3f} s ({len(resultado)} filas de resultado)")
    return duracion

tiempo_parquet = benchmark(df_parquet, "Parquet")
tiempo_csv = benchmark(df_csv, "CSV")

ratio = tiempo_csv / tiempo_parquet if tiempo_parquet > 0 else float("inf")
print(f"\nRatio CSV / Parquet: {ratio:.2f}x")

Parquet: 1.696 s (6 filas de resultado)

CSV: 0.817 s (6 filas de resultado)

Ratio CSV / Parquet: 0.48x

![ejec5](ejecucion5.png)

## Análisis — completa antes de entregar

### a) Tamaño en disco

¿Cuánto pesa `prueba_parquet.parquet` frente a `prueba_csv.csv`? (revisa
la salida de `generar_datos.py`, o `aws s3 ls --human-readable` sobre
`bronze/ventas/`).

→ → Parquet pesa 185.5 KiB y CSV 798.3 KiB. CSV ocupa aproximadamente 4.3 veces más espacio que Parquet. Parquet ocupa menos espacio porque es un formato columnar y utiliza compresión Snappy.

### b) Tiempo de la consulta

¿Cuánto tardó la consulta de la Celda 3 en cada formato?

→ La consulta tardó 1.696 s usando Parquet y 0.817 s usando CSV. En esta ejecución, CSV fue más rápido para la consulta realizada.

### c) Ratio de mejora

¿Cuál fue el ratio de mejora (CSV / Parquet) que obtuviste en la Celda 3?
¿Coincide con el orden de magnitud visto en clase (~9x)? Si es distinto,
¿a qué le atribuyes la diferencia (tamaño del clúster, tamaño del
dataset, tipo de consulta)?

→ El ratio obtenido fue 0.48x, calculado como 0.817 / 1.696. Esto significa que CSV tardó aproximadamente la mitad del tiempo que Parquet en esta ejecución, por lo que el resultado no coincide con la mejora aproximada de 9x observada en clase. La diferencia puede deberse al tamaño reducido del dataset, al clúster pequeño, al costo inicial de lectura de Parquet y a factores variables de Spark, S3 y la JVM. Además, con solo 10.000 filas, los costos fijos de inicialización pueden ser más importantes que las ventajas de Parquet.

### d) Conexión con el Teorema CAP

¿Por qué S3 con replicación entre múltiples zonas de disponibilidad es
una decisión **CP** dentro del Teorema CAP? ¿Qué está sacrificando S3 a
cambio de esa garantía de consistencia?

→ S3 puede considerarse una decisión CP en el contexto de este laboratorio porque prioriza la consistencia de los datos frente a una partición de red. S3 confirma una escritura cuando el objeto está almacenado de forma durable y, después de una escritura exitosa, una lectura obtiene la versión más reciente.

Ante una partición o un problema de comunicación entre zonas, el sistema puede rechazar o retrasar algunas operaciones antes que devolver una versión desactualizada del objeto. Por tanto, la propiedad que se sacrifica es la disponibilidad, especialmente la posibilidad de aceptar lecturas o escrituras durante una interrupción. La tolerancia a particiones se mantiene, ya que es la condición asumida por el Teorema CAP.

## Captura del DAG en Spark UI

1. En EMR Studio (o en la consola de tu clúster), abre **Spark UI /
   History Server**.
2. Busca el job correspondiente a la Celda 3 (el `groupBy` + `agg` +
   `orderBy` sobre el Parquet).
3. Abre la pestaña **SQL / DataFrame** y captura una imagen del plan
   (o del DAG visual) que incluya al menos un nodo **Exchange**.
4. Guarda la captura como `dag_spark_ui.png` dentro de tu carpeta de
   entrega y referencíala en tu PR.

**Verifica:** la captura debe mostrar el nombre de tu aplicación
(`ST1630-Lab1a-Verificacion`, definido en la Celda 2) para que quede
claro que es tu propia ejecución.

![dag_spark_ui](dag_spark_ui.png)


## Bitácora de delegación

Completa según lo que realmente delegaste a un agente de IA durante
este notebook (ver `../../../docs/politica-ia.md`).

| Tarea | ¿Delegado a agente? | Herramienta | Justificación |
|---|---|---|---|
| Boilerplate de SparkSession / lectura de S3 | → No | → [] | → El codigo ya estaba en el notebook; yo solo edite la variable bucket. |
| Diseño de la consulta del benchmark (Celda 3) | → No | → [] | → La consulta ya estaba implementada en el notebook |
| Interpretación de los resultados (Celda 4) | → Parcial | → Copilot Student | → Utilize IA como apoyo para entender y organizar los resultados obtenidos, pero las respuestas se basaron en mi entendimiento de la ayuda que me daba la IA. |
| Troubleshooting de errores de conexión a S3 | → Sí | → Gemini-Copilot Student | → Se utilizó IA para identificar y solucionar problemas de configuración de AWS CLI, IAM, EMR, roles, EMR Studio y conexión del clúster, más que todo porque no pude usar EMR studio, entonces la IA me ayudo a implementarlo con SSH |

> Recuerda: la interpretación de los resultados y la conexión con CAP
> (pregunta d) deben reflejar tu propio razonamiento — ver
> `../README.md`, sección "Bitácora de delegación".